In [18]:
import sys
import os

current_file_path = os.path.abspath(os.getcwd()) # Funciona solo para JupyterNotebooks

# Obtener la ruta del directorio raíz del proyecto
project_root = os.path.abspath(os.path.join(os.path.dirname(current_file_path)))

# Añadir el directorio raíz del proyecto al sys.path
sys.path.append(project_root)

from config import INPUTS_FOLDER
import os

statement_path = os.path.join(INPUTS_FOLDER, 'test_files', 'Banorte', 'Banorte_debit_statement.pdf')
statement_path

'c:\\Proyectos\\Aether\\Aether_v1\\documents\\inputs\\test_files\\Banorte\\Banorte_debit_statement.pdf'

In [19]:
from document_reader import PDFReader

reader = PDFReader(statement_path)
extracted_words = reader.extract_words_from_pdf()

extracted_words

,page,page_height,text,x0,top,x1,bottom
0,1,792.0,MARIA,48.816000,51.69000,67.446000,60.6900
1,1,792.0,FERNANDA,68.859000,51.69000,98.856000,60.6900
2,1,792.0,CANUDAS,100.269000,51.69000,126.882000,60.6900
3,1,792.0,ZERTUCHE,128.295000,51.69000,156.294000,60.6900
4,1,792.0,CALLE,48.816000,59.40000,65.520000,68.4000
...,...,...,...,...,...,...,...
1853,6,792.0,6/6,557.280000,764.60400,566.739000,773.6040
1854,6,792.0,Nuevo,50.400000,772.49955,69.216588,780.4998
1855,6,792.0,Leon.,70.688634,772.49955,86.553130,780.4998
1856,6,792.0,RFC,88.025176,772.49955,99.017519,780.4998


In [20]:
from bank_detector import DefaultBankDetector

bank_detector = DefaultBankDetector(extracted_words)

bank = bank_detector.detect_bank()
statement_type = bank_detector.detect_statement_type()
statement_properties = bank_detector.statement_propertys

print(f'{bank} - {statement_type}')

banorte - debit


In [21]:
from table_boundary_detector import TransactionTableBoundaryDetector

boundary_detector = TransactionTableBoundaryDetector(extracted_words, statement_properties)

start_idx = boundary_detector.start_idx
end_idx = boundary_detector.end_idx

df_table = boundary_detector.get_filtered_table_words()

print(f'{start_idx} - {end_idx}')
df_table

441 - 936


,page,page_height,text,x0,top,x1,bottom
0,4,792.0,Cuenta,269.99550,68.946,292.10850,77.946
1,4,792.0,Enlace,293.53950,68.946,313.70850,77.946
2,4,792.0,Personal,315.13950,68.946,342.22050,77.946
3,4,792.0,FECHA,59.04375,84.780,78.42975,93.780
4,4,792.0,DESCRIPCIÓN,87.07425,84.780,127.47525,93.780
...,...,...,...,...,...,...,...
490,4,792.0,las,320.43525,589.080,328.03125,598.080
491,4,792.0,13:53,329.44425,589.080,345.05025,598.080
492,4,792.0,hrs.,85.05825,599.880,95.16525,608.880
493,4,792.0,-,96.57825,599.880,99.08925,608.880


In [22]:
from row_segmenter import TransactionRowSegmenter

row_segmenter = TransactionRowSegmenter(df_table, statement_properties)

header_row = row_segmenter.get_header_row()
grouped_rows = row_segmenter.group_rows()

print(header_row)
grouped_rows

{'columns': ['FECHA', 'DESCRIPCIÓN / ESTABLECIMIENTO', 'MONTO DEL DEPOSITO', 'MONTO DEL RETIRO', 'SALDO'], 'x0': [59.043749999999996, 87.07425, 353.55225, 431.03774999999996, 533.2755], 'x1': [78.42975, 188.30625, 420.18825000000004, 489.84374999999994, 553.1114999999999]}


,row_group,text,words,top,bottom,page
0,0,Cuenta Enlace Personal,"[(Cuenta, 269.9955, 292.1085), (Enlace, 293.53...",68.946,77.946,4
1,1,FECHA DESCRIPCIÓN / ESTABLECIMIENTO MONTO DEL ...,"[(FECHA, 59.043749999999996, 78.42975), (DESCR...",84.780,93.780,4
2,2,03-FEB-24 SALDO ANTERIOR 157.39,"[(03-FEB-24, 54.27825, 103.33725), ( SALDO ANT...",98.130,107.130,4
3,3,06-FEB-24 TRASP FONDOS 0000240206 AL R.F.C. C...,"[(06-FEB-24, 54.27825, 102.73424999999999), ( ...",109.680,129.480,4
4,4,"08-FEB-24 TRASPASO 0000240208 , DE LA CUENTA:...","[(08-FEB-24, 54.27825, 113.39925), ( TRASPASO ...",132.030,141.030,4
5,5,08-FEB-24 TRASP FONDOS 0000240208 AL R.F.C. G...,"[(08-FEB-24, 54.27825, 102.73424999999999), ( ...",143.580,163.380,4
6,6,13-FEB-24 Deposito en efectivo en cajero Bano...,"[(13-FEB-24, 54.27825, 109.36725000000001), ( ...",165.930,196.530,4
7,7,15-FEB-24 Deposito en efectivo en cajero Bano...,"[(15-FEB-24, 54.27825, 109.36725000000001), ( ...",199.080,229.680,4
8,8,"15-FEB-24 TRASPASO 0000240215 , DE LA CUENTA:...","[(15-FEB-24, 54.27825, 113.39925), ( TRASPASO ...",232.230,241.230,4
9,9,15-FEB-24 2024021640014TRAP0000466719370 SPEI...,"[(15-FEB-24, 54.27825, 189.57525000000004), ( ...",243.780,285.180,4


In [23]:
from table_reconstructor import TransactionTableReconstructor

table_reconstructor = TransactionTableReconstructor(grouped_rows, header_row, statement_properties)
df_table = table_reconstructor.get_structured_table()
df_table

,FECHA,DESCRIPCIÓN / ESTABLECIMIENTO,MONTO DEL DEPOSITO,MONTO DEL RETIRO,SALDO
0,03-FEB-24,SALDO ANTERIOR,,,157.39
1,06-FEB-24,TRASP FONDOS 0000240206 AL R.F.C. CENE7504044L...,,20.00,137.39
2,08-FEB-24,"TRASPASO 0000240208 , DE LA CUENTA: 1256031099",700.00,,837.39
3,08-FEB-24,TRASP FONDOS 0000240208 AL R.F.C. GOGC880322M1...,,780.00,57.39
4,13-FEB-24,Deposito en efectivo en cajero Banorte. Te dep...,"1,600.00",,"1,657.39"
5,15-FEB-24,Deposito en efectivo en cajero Banorte. Te dep...,"4,050.00",,"5,707.39"
6,15-FEB-24,"TRASPASO 0000240215 , DE LA CUENTA: 1256031099","3,801.50",,"9,508.89"
7,15-FEB-24,"2024021640014TRAP0000466719370 SPEI RECIBIDO, ...",500.00,,"10,008.89"
8,15-FEB-24,CARGO POR PAGO CONCENTRACION MOV TARJETA DE CR...,,"9,735.58",273.31
9,19-FEB-24,FAR GUAD 2413 RFC:FGU 830930PD3,,28.00,245.31


In [24]:
from table_normalizer import TransactionTableNormalizer

normalizer = TransactionTableNormalizer(df_table, extracted_words, statement_properties)

normalizer.normalize_table() # Final result

,Date,Description,Amount,Type
0,2024-02-03,SALDO ANTERIOR,0.00,Saldo
1,2024-02-06,TRASP FONDOS 0000240206 AL R.F.C. CENE7504044L...,-20.00,Cargo
2,2024-02-08,"TRASPASO 0000240208 , DE LA CUENTA: 1256031099",700.00,Abono
3,2024-02-08,TRASP FONDOS 0000240208 AL R.F.C. GOGC880322M1...,-780.00,Cargo
4,2024-02-13,Deposito en efectivo en cajero Banorte. Te dep...,1600.00,Abono
5,2024-02-15,Deposito en efectivo en cajero Banorte. Te dep...,4050.00,Abono
6,2024-02-15,"TRASPASO 0000240215 , DE LA CUENTA: 1256031099",3801.50,Abono
7,2024-02-15,"2024021640014TRAP0000466719370 SPEI RECIBIDO, ...",500.00,Abono
8,2024-02-15,CARGO POR PAGO CONCENTRACION MOV TARJETA DE CR...,-9735.58,Cargo
9,2024-02-19,FAR GUAD 2413 RFC:FGU 830930PD3,-28.00,Cargo
